In [12]:
import pandas as pd
df_records = pd.read_csv("student_academic_records.csv")

In [13]:
# Filter for 2024 cohort
df_2024 = df_records[df_records["EntryYear"] == 2024]

# Count only ORIGINAL attempts (not resits)
module_counts = (
    df_2024[df_2024["AttemptType"] == "Original"]
    .groupby("StudentID")
    .size()
)

# Find students with exactly 4 modules completed
students_with_4 = module_counts[module_counts == 4].index.tolist()

students_with_4

[20240017, 20240068, 20240099, 20240104, 20240136, 20240146]

In [24]:
def find_2021_students_at_risk(df_records):
    """
    Returns a DataFrame of 2021 students who are at risk of academic probation.
    Risk criteria:
      - Cumulative GPA < 2.0
      - OR at least one failing grade (D/F)
      - OR at least one resit
    """

    # Filter for 2021 cohort
    df_2021 = df_records[df_records["EntryYear"] == 2021]

    at_risk = []

    # Group by student
    for sid, sdf in df_2021.groupby("StudentID"):

        # Compute cumulative GPA using only IncludedInGPA rows
        gpa_df = sdf[sdf["IncludedInGPA"] == True]

        if len(gpa_df) > 0:
            cum_gpa = (gpa_df["GradePoints"] * gpa_df["Credits"]).sum() / gpa_df["Credits"].sum()
        else:
            cum_gpa = 0

        # Risk indicators
        has_fail = any(sdf["Grade"].isin(["D", "F"]))
        has_resit = any(sdf["AttemptType"] == "Resit")

        # Risk condition
        if cum_gpa < 2.0 or has_fail or has_resit:
            at_risk.append({
                "StudentID": sid,
                "CumulativeGPA": round(cum_gpa, 2),
                "FailedModules": sum(sdf["Grade"].isin(["D", "F"])),
                "Resits": sum(sdf["AttemptType"] == "Resit")
            })

    return pd.DataFrame(at_risk)


In [25]:
find_2021_students_at_risk(df_records)

,StudentID,CumulativeGPA,FailedModules,Resits
0,20210001,2.81,2,2
1,20210007,2.62,3,3
2,20210008,2.69,1,1
3,20210013,2.78,3,3
4,20210014,2.62,1,1
5,20210018,2.88,2,2
6,20210025,2.00,3,3
7,20210028,2.66,3,3
8,20210034,2.75,2,2
9,20210036,2.56,1,1


In [26]:
at_risk_2021 = find_2021_students_at_risk(df_records)

In [6]:
def compute_student_gpa(df_student):
    """
    Computes semester GPA and cumulative GPA for a single student's full record.
    Respects:
      - D/F = fail
      - Resits
      - Grade forgiveness
      - IncludedInGPA flag
    """
    df_student = df_student.sort_values(["Year", "Semester"])

    results = []

    cumulative_points = 0
    cumulative_credits = 0

    for (year, sem), sem_df in df_student.groupby(["Year", "Semester"]):

        # Only include attempts marked as IncludedInGPA
        gpa_df = sem_df[sem_df["IncludedInGPA"] == True]

        # Semester GPA
        if len(gpa_df) > 0:
            sem_points = (gpa_df["GradePoints"] * gpa_df["Credits"]).sum()
            sem_credits = gpa_df["Credits"].sum()
            sem_gpa = sem_points / sem_credits
        else:
            sem_points = 0
            sem_credits = 0
            sem_gpa = 0

        # Update cumulative totals
        cumulative_points += sem_points
        cumulative_credits += sem_credits

        # Cumulative GPA
        if cumulative_credits > 0:
            cumulative_gpa = cumulative_points / cumulative_credits
        else:
            cumulative_gpa = 0

        results.append({
            "Year": year,
            "Semester": sem,
            "SemesterGPA": round(sem_gpa, 3),
            "CumulativeGPA": round(cumulative_gpa, 3),
            "FailedModules": sum(sem_df["Grade"].isin(["D", "F"])),
            "Resits": sum(sem_df["AttemptType"] == "Resit"),
            "Forgiven": sum(sem_df["Forgiven"] == True)
        })

    return pd.DataFrame(results)


In [10]:
def print_student_grades(df_records, student_id):
    """
    Prints all grade details for a given student in a clean, readable format.
    Includes:
      - Original attempts
      - Resits
      - Forgiveness
      - GPA inclusion
      - Grade points
      - Year + Semester
    """

    sdf = df_records[df_records["StudentID"] == student_id]

    if sdf.empty:
        print(f"No records found for student {student_id}")
        return

    sdf = sdf.sort_values(["Year", "Semester", "Module", "AttemptType"])

    print(f"\n=== Grade Details for Student {student_id} (Entry Year {sdf['EntryYear'].iloc[0]}) ===\n")

    for (year, sem), group in sdf.groupby(["Year", "Semester"]):
        print(f"--- Year {year}, Semester {sem} ---")

        for _, row in group.iterrows():
            print(
                f"Module: {row['Module']}\n"
                f"  Attempt: {row['AttemptType']}\n"
                f"  Grade: {row['Grade']}  (Points: {row['GradePoints']})\n"
                f"  Included in GPA: {row['IncludedInGPA']}\n"
                f"  Forgiven: {row['Forgiven']}\n"
            )


In [21]:
#version 2
def print_student_grades(df_records, student_id):
    """
    Prints all grade details for a student in a compact tabular format.
    GPA-included attempts appear first.
    Resit attempts appear as auxiliary rows.
    """

    sdf = df_records[df_records["StudentID"] == student_id]

    if sdf.empty:
        print(f"No records found for student {student_id}")
        return

    sdf = sdf.sort_values(
        ["Year", "Semester", "Module", "IncludedInGPA", "AttemptType"],
        ascending=[True, True, True, False, True]
    )

    print(f"\n=== Grade Report for Student {student_id} (Entry Year {sdf['EntryYear'].iloc[0]}) ===\n")

    for (year, sem), group in sdf.groupby(["Year", "Semester"]):
        print(f"--- Year {year}, Semester {sem} ---")

        # Build a compact table
        table_rows = []
        for _, row in group.iterrows():
            table_rows.append([
                row["Module"],
                row["AttemptType"],
                row["Grade"],
                row["GradePoints"],
                "Yes" if row["IncludedInGPA"] else "No",
                "Yes" if row["Forgiven"] else "No"
            ])

        # Print header
        print(f"{'Module':<10} {'Attempt':<10} {'Grade':<6} {'Pts':<4} {'InGPA':<6} {'Forgiven':<8}")
        print("-" * 52)

        # Print rows
        for r in table_rows:
            print(f"{r[0]:<10} {r[1]:<10} {r[2]:<6} {r[3]:<4} {r[4]:<6} {r[5]:<8}")

        print()  # blank line after each semester


In [22]:

student_id = 20240017
print_student_grades(df_2024, student_id)


=== Grade Report for Student 20240017 (Entry Year 2024) ===

--- Year 1, Semester 1 ---
Module     Attempt    Grade  Pts  InGPA  Forgiven
----------------------------------------------------
Dbase      Original   B+     3.5  Yes    No      
Eng1       Original   B      3.0  Yes    No      
IT         Original   F      0.0  Yes    No      
IT         Resit      A      4.0  No     No      
Maths1     Original   B+     3.5  Yes    No      



In [ ]:
print(df_2024)

In [23]:

# Compute GPA for one student
student_id = 20240017
df_student = df_records[df_records["StudentID"] == student_id]

gpa_table = compute_student_gpa(df_student)
print(gpa_table)

   Year  Semester  SemesterGPA  CumulativeGPA  FailedModules  Resits  Forgiven
0     1         1          2.5            2.5              1       1         0


In [29]:
print_student_grades(df_records, 20210103)


=== Grade Report for Student 20210103 (Entry Year 2021) ===

--- Year 1, Semester 1 ---
Module     Attempt    Grade  Pts  InGPA  Forgiven
----------------------------------------------------
Dbase      Original   C+     2.5  Yes    No      
Eng1       Original   D      1.0  Yes    No      
Eng1       Resit      C+     2.5  No     No      
IT         Original   A      4.0  Yes    No      
Maths1     Original   B+     3.5  Yes    No      

--- Year 1, Semester 2 ---
Module     Attempt    Grade  Pts  InGPA  Forgiven
----------------------------------------------------
Eng2       Original   C      2.0  Yes    No      
Maths2     Original   F      0.0  Yes    No      
Maths2     Resit      B      3.0  No     No      
Network    Original   C+     2.5  Yes    No      
SoftEng    Original   A      4.0  Yes    No      

--- Year 2, Semester 1 ---
Module     Attempt    Grade  Pts  InGPA  Forgiven
----------------------------------------------------
AI1        Original   D      1.0  Yes    No   